# 01. 데이터 이해 및 진단

평가항목 1(데이터 이해 및 진단)의 근거를 그림과 표로 정리하는 노트북입니다.

- 정제 로직은 `src/data.py`, 15분 변환과 피처는 `src/features.py`에서 불러옵니다. 이 노트북에는 확인과 시각화 코드만 둡니다.
- 실행 전에 원본 CSV를 `data/okm_augumented_2021.csv`에 두세요.
- 수정한 뒤에는 항상 **커널 재시작 → 전체 실행**이 통과하는지 확인하세요.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import config as C
from src.data import load_raw, clean
from src.features import to_long, build_features, feature_columns
from src.viz import setup_plot_style, heatmap, DOW_KR

setup_plot_style()
pd.set_option("display.max_columns", 50)
DATA = ROOT / C.DATA_PATH
print("데이터 경로:", DATA)

## 1. 원본 데이터 개요

In [ ]:
raw = load_raw(DATA)
print("행, 열:", raw.shape)
display(raw.drop(columns="_row").head())
display(raw.drop(columns="_row").describe().T.round(2))

| 컬럼 | 해석 | 근거 |
|---|---|---|
| `날짜`, `시간` | 생산 단위는 **1시간**, 하루 24행 | 2절 |
| `15분`, `30분`, `45분`, `60분` | 각 15분 구간의 평균수요전력(kW), 해석 A | 가이드북 설명과 한전 15분 수요 기준 |
| `평균` | 네 값의 평균(반올림), 해석 A에서 시간당 kWh와 같음 | 3절 |
| `생산량` | ERP에 등록된 시간별 생산량 | 가이드북 |
| `기온`, `풍속`, `습도`, `강수량` | 기상 | |
| `전기요금(계절)` | 월별 상수(계절 요금) | 4절 |
| `day`, `d`, `m` | 요일(월=1), 일, 월 | 3절 |
| `공장인원` | **인원이 아님.** 생산량 ÷ 전력 합계로 계산된 파생변수 | 4절 |
| `인건비` | 시간대 가중치(09~17시 1.0, 그 외 1.5) | 4절 |

## 2. 생산 단위와 시간 구조

날짜마다 정확히 24행인지, `시간` 컬럼이 0~23 범위인지 확인합니다.

In [ ]:
per_day = raw.groupby("날짜").size()
print("날짜별 행 수 분포:", per_day.value_counts().to_dict())
print("시간 컬럼 범위:", raw["시간"].min(), "~", raw["시간"].max())
bad = raw[raw["시간"] > 23]
print(f"0~23을 벗어난 행: {len(bad)}개, 날짜: {sorted(bad['날짜'].unique().tolist())}")
display(raw[raw["날짜"] == 20210713].drop(columns="_row").head(6))

7/13, 7/15는 `시간` 컬럼에 전력값처럼 보이는 숫자가 들어 있습니다. 네 전력값과 `평균`은 서로 맞으므로 전력 데이터는 정상이고 `시간`만 오염된 것으로 판단합니다. 날짜별로 정확히 24행이므로 **행 순서로 시간을 복원**합니다(`clean()`의 1단계).

In [ ]:
hourly, dq, groups = clean(raw)
long = to_long(hourly)   # 15분 단위 시계열 (이후 절에서 사용)
print("복원한 행:", dq["hour_repaired"])
print("복원 후 결측 시각:", dq["missing_hours_after_repair"], "/ 중복 시각:", dq["duplicate_ts_after_repair"])
display(hourly[hourly["date"] == "2021-07-13"].head(6))

## 3. 정합성 검증

In [ ]:
mean4 = raw[C.POWER_COLS].mean(axis=1)
diff = (raw["평균"] - mean4).round(2)
print("평균 − (네 값의 평균) 분포:", diff.value_counts().sort_index().to_dict())
print("day 컬럼 = 실제 요일(월=1):", dq["day_col_is_weekday_mon1_sun7"])

`평균`과 네 값의 평균 차이는 반올림 범위(±0.5) 안에만 있습니다. 즉 `평균`은 네 15분 값의 평균이고, 해석 A에서는 그 시간의 사용량(kWh)과 숫자상 같습니다.

## 4. 변수 의미 검증

### 4-1. `공장인원`은 전력으로 계산된 파생변수

In [ ]:
s = raw[C.POWER_COLS].sum(axis=1)
ratio = raw["생산량"] / s.where(s > 0)
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(ratio, raw["공장인원"], s=4, alpha=0.5)
lim = [0, np.nanmax(ratio) * 1.05]
ax.plot(lim, lim, "r--", lw=1, label="y = x")
ax.set_xlabel("생산량 ÷ (15분+30분+45분+60분)")
ax.set_ylabel("공장인원")
ax.set_title("공장인원과 계산값이 완전히 일치")
ax.legend()
plt.show()
print("최대 오차:", dq["leakage_factory_staff"]["max_abs_diff_vs_prod_over_power_sum"])
print("결측 행(0/0):", dq["leakage_factory_staff"]["nan_rows_all_0_div_0"])

`공장인원`은 타깃(전력)을 분모로 써서 계산된 값이라, 피처로 쓰면 정답이 새어 들어갑니다(누수). 정제 단계에서 제외합니다.

### 4-2. `인건비`는 시간대 규칙으로 결정되고, 오염된 날은 잘못 계산돼 있음

In [ ]:
rawh = raw.sort_values(["날짜", "_row"]).assign(hour=lambda d: d.groupby("날짜").cumcount())
rule = np.where(rawh["hour"].between(9, 17), 1.0, 1.5)
exc = rawh[rawh["인건비"] != rule]
print(f"규칙(09~17시 1.0, 그 외 1.5)의 예외: {len(exc)}행, 날짜: {sorted(exc['날짜'].unique().tolist())}")
lab = hourly.pivot_table(index="hour", columns="dow", values="labor_mult", aggfunc="mean").rename(columns=DOW_KR)
fig, ax = plt.subplots(figsize=(5, 6))
heatmap(lab, ax=ax, cmap="Blues", annotate=True, fmt="{:.1f}", cbar_label="인건비 가중치 (정제 후)")
ax.set_xlabel("요일")
ax.set_ylabel("시간")
plt.show()

예외는 `시간`이 오염된 7/13, 7/15뿐입니다. 오염된 `시간` 값(모두 17보다 큼)으로 인건비를 계산했기 때문에 주간에도 1.5가 들어간 것으로 보입니다. 정제 단계에서 복원한 시간으로 다시 계산합니다. 인건비는 시간만으로 결정되는 값이라 예측 피처로는 쓰지 않고, 피크저감 비용 계산에 사용합니다.

### 4-3. `전기요금(계절)`은 월별 상수

In [ ]:
display(raw.groupby("m")["전기요금(계절)"].agg(["min", "max", "nunique"]))

## 5. 결측과 전력 0 구간

In [ ]:
print("결측이 있는 컬럼:", raw.isna().sum()[lambda x: x > 0].to_dict())
print("전력 0 (시간 단위, 네 칸 모두 0):", dq["zero_power_hours"])
print("전력 0 (15분 단위):", dq["zero_power_15min_slots"])
z = long[(long["date"] >= "2021-08-27") & (long["date"] <= "2021-08-30")]
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(z["ts"], z["kw"], lw=1)
ax.set_ylabel("15분 수요 (kW)")
ax.set_title("8/28~29 전력 0 구간 (정전·셧다운 추정)")
plt.show()

`공장인원`의 결측 17개는 생산량과 전력이 모두 0인 0/0입니다. 기상 결측은 시간순 선형보간했습니다.

정전은 8/28 17:30에 시작해 시간 중간에 걸쳐 있어서, 시간 단위 판정(네 칸 모두 0)으로는 시작·종료 부분 칸을 놓칩니다. 그래서 **15분 단위(전력이 정확히 0인 칸)로 판정**합니다. 정전 칸은 삭제하지 않고 표시만 해두고, 학습에서는 제외, 평가에서는 포함·제외 두 가지로 보며, 래그 계산에서는 대체값을 씁니다(8절).

## 6. 증강 복제일

하루치 96개 값(24시간 × 4)이 완전히 같은 날을 같은 복제그룹으로 묶었습니다.

In [ ]:
print({k: v for k, v in dq["duplicate_days"].items() if k != "days_in_duplicate_groups_by_month"})
display(groups.head(8))

d = hourly.groupby("date").agg(size=("dup_group_size", "first"))
by_m = pd.DataFrame({
    "고유한 날": (d["size"] == 1).groupby(d.index.month).sum(),
    "복제그룹 소속": (d["size"] > 1).groupby(d.index.month).sum(),
})
ax = by_m.plot(kind="bar", stacked=True, figsize=(8, 3.5))
ax.set_xlabel("월")
ax.set_ylabel("일수")
ax.set_title("복제일은 1~6월에 집중, 8~9월은 전부 고유")
ax.legend(loc="upper left", bbox_to_anchor=(1.0, 1.0))
plt.show()

가동일 패턴(하루 평균 50kW 초과)을 가진 가장 큰 복제그룹을 겹쳐 그려 보면 선이 완전히 하나로 겹칩니다. 전력은 복제됐지만 그 날의 요일·기상·생산량은 서로 다릅니다.

In [ ]:
group_kw = long.groupby("dup_group")["kw"].mean()
g = groups[groups["dup_group"].map(group_kw) > 50].iloc[0]
dates = pd.to_datetime(g["dates"].split(", "))
fig, ax = plt.subplots(figsize=(12, 3.5))
for dt in dates:
    s = long[long["date"] == dt]
    ax.plot(s["slot"], s["kw"], lw=1, label=f"{dt:%m-%d} ({DOW_KR[dt.dayofweek + 1]})")
ax.set_xlabel("15분 슬롯 (0~95)")
ax.set_ylabel("15분 수요 (kW)")
ax.legend(ncol=4, fontsize=8)
ax.set_title("같은 복제그룹의 전력 프로파일 (완전히 겹침)")
plt.show()
ctx = hourly[hourly["date"].isin(dates)].groupby("date").agg(
    요일=("dow", lambda x: DOW_KR[x.iloc[0]]), 생산량_합계=("prod", "sum"), 평균기온=("temp", "mean"))
display(ctx.round(1))

복제된 날은 전력과 그 날의 맥락(요일·생산량·달력)이 어긋날 수 있습니다. 하루 평균 50kW 초과를 "전력상 가동"으로 보고 생산량 유무, 달력(평일·비공휴일)과 비교합니다. 토요일은 새벽 야간가동만 있어 달력 비교에서 제외했습니다.

In [ ]:
day = (long.groupby("date")
       .agg(kw=("kw", "mean"), prod=("prod", "sum"), dow=("dow", "first"),
            is_copy=("is_copy_day", "first"), size=("dup_group_size", "first")))
day["prod"] = day["prod"] / 4   # 15분 행마다 시간 생산량이 복사돼 있으므로 되돌림
hol = pd.to_datetime(C.HOLIDAYS)
on_power = day["kw"] > 50
on_cal = (day["dow"] <= 5) & ~day.index.isin(hol)
day["유형"] = np.select([day["size"] == 1, day["is_copy"]], ["고유한 날", "복제본(2번째 이후)"], "복제그룹 첫 등장일")
day["생산량과 불일치"] = (on_power != (day["prod"] > 0)).astype(float)
day["달력과 불일치"] = (on_power != on_cal).astype(float).where(day["dow"] != 6)
display(day.groupby("유형").agg(일수=("kw", "size"), 생산량과_불일치율=("생산량과 불일치", "mean"),
                              달력과_불일치율=("달력과 불일치", "mean")).round(3))
display(day[(day["유형"] != "고유한 날") & (day["생산량과 불일치"] == 1)]
        [["kw", "prod", "dow", "유형"]].round(1).head(8))

불일치는 복제본에만 있는 것이 아니라 고유한 날에도 있습니다. 그래서 복제 여부만으로 학습 데이터를 거르는 것(`COPY_POLICY="drop"`)은 원인을 정확히 겨냥하지 못하며, CV에서도 `keep`과 차이가 노이즈 수준이었습니다. 복제일로 인한 **누수**는 학습 정책과 별개로, 검증 단계의 복제그룹 제거(purge)와 원본일 채점으로 막습니다.

## 7. 전력 사용 패턴 (시간·요일·계절)

복제본과 정전 구간을 제외한 날만으로 봅니다.

In [ ]:
ok = long[~long["zero_power"] & ~long["is_copy_day"]]
hm = ok.pivot_table(index="hour", columns="dow", values="kw", aggfunc="mean").rename(columns=DOW_KR)
fig, ax = plt.subplots(figsize=(6, 8))
heatmap(hm, ax=ax, cmap="magma", annotate=True, cbar_label="평균 15분 수요 (kW)")
ax.set_xlabel("요일")
ax.set_ylabel("시간")
ax.set_title("시간 × 요일 평균 수요")
plt.show()

평일 주간(08~19시) 가동, 12시 점심 급감, 월요일 밤~토요일 아침의 야간 가동, 일요일 대기부하 구조가 보입니다.

In [ ]:
dmax = ok.loc[ok.groupby("date")["kw"].idxmax()]
weekday_max = dmax[dmax["dow"] <= 5]
fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
weekday_max["hour"].value_counts().sort_index().plot(kind="bar", ax=axes[0])
axes[0].set_xlabel("일 최대수요 발생 시간 (평일)")
axes[0].set_ylabel("일수")
axes[0].set_title("일 최대수요는 08시 기동과 오전에 집중")
mon = ok.groupby(ok["ts"].dt.month).agg(kw=("kw", "mean"), temp=("temp", "mean"))
axes[1].bar(mon.index, mon["kw"], color="#8aa")
axes[1].set_xlabel("월")
axes[1].set_ylabel("평균 15분 수요 (kW)")
ax2 = axes[1].twinx()
ax2.plot(mon.index, mon["temp"], "o-", color="tab:red")
ax2.set_ylabel("평균 기온 (°C)")
axes[1].set_title("월별 평균 수요와 기온")
plt.tight_layout()
plt.show()

## 8. 15분 변환과 피처 확인

예측 과제는 **D일 00:00 시점에 D일 96개 15분 슬롯을 예측**하는 것이고, 피처는 그 시점에 알 수 있는 정보만 씁니다. 예시로 2021-08-30(월) 08:15 슬롯의 피처 값을 봅니다.

In [ ]:
feat = build_features(long)
cols = feature_columns()
print(f"{len(feat):,}행 (첫 7일은 lag_7d가 없어 제외), 피처 {len(cols)}개")
example = feat[feat["ts"] == "2021-08-30 08:15"]
display(example[["ts", "kw", *cols]].T.rename(columns=lambda _: "값"))

전날(8/29) 같은 칸은 정전이었으므로, `lag_1d`와 전날 통계는 정전 칸을 **같은 요일·같은 칸의 최근 4주 중앙값**으로 대체한 시계열에서 계산됩니다(`features.lag_source`). 타깃(`kw`)은 원래 값 그대로입니다. 아래는 대체 전후의 래그 비교입니다.

In [ ]:
from src.features import lag_source
src_series = lag_source(long)
outage = long["zero_power"]
cmp = pd.DataFrame({"ts": long.loc[outage, "ts"], "원래 kw": long.loc[outage, "kw"], "래그용 대체값": src_series[outage]})
display(cmp.head(8))
fig, ax = plt.subplots(figsize=(12, 3))
w = (long["ts"] >= "2021-08-27") & (long["ts"] < "2021-08-31")
ax.plot(long.loc[w, "ts"], long.loc[w, "kw"], lw=1, label="원래 kw (타깃)")
ax.plot(long.loc[w, "ts"], src_series[w], lw=1, ls="--", label="래그 계산용 시계열")
ax.set_ylabel("15분 수요 (kW)")
ax.legend(fontsize=8)
ax.set_title("정전 칸은 래그 계산에서만 대체")
plt.show()

## 9. 진단 요약

| 문제 | 검증 근거 | 처리 |
|---|---|---|
| 7/13, 7/15의 `시간` 오염 | 날짜별 24행, 복원 후 결측·중복 시각 0 | 행 순서로 복원 |
| 같은 날의 `인건비` 오류 | 나머지 행에서 시간대 규칙이 예외 없이 성립 | 복원된 시간으로 재계산 |
| `공장인원` | 생산량 ÷ 전력 합계와 오차 1e-8 이내 일치 | 누수로 제외 |
| `인건비`, `전기요금(계절)` | 시간대·월로 완전히 결정 | 피처 제외, 비용 계산용 |
| 기상 결측 4개 | | 선형보간 |
| 전력 0 (정전) | 15분 단위 74칸 (시간 단위 판정은 6칸 누락) | 15분 단위 판정, 학습 제외, 평가는 포함·제외 모두, 래그는 최근 4주 같은 요일·칸 중앙값으로 대체 |
| 증강 복제일 160일 | 96포인트 해시 일치, 1~6월 집중 | 검증에서 purge·원본일 채점 |

정제 과정의 모든 검증 수치는 아래와 같고, `python run.py` 실행 시 `outputs/data_quality.json`에도 저장됩니다.

In [ ]:
import json
print(json.dumps(dq, ensure_ascii=False, indent=2, default=str))